In [16]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("data.csv")

# Remove rows with missing target values
df = df.dropna(subset=["TenYearCHD"])

X = df.drop(columns=["TenYearCHD"])
y = df["TenYearCHD"].astype(int)

# Save feature names
columns = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Handle missing values, scale features, and train Logistic Regression
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Save model and columns
with open("heart_disease_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("heart_disease_columns.pkl", "wb") as f:
    pickle.dump(columns, f)

print("\nModel trained and saved successfully!")

Accuracy: 0.847877358490566

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.99      0.92       719
           1       0.50      0.07      0.12       129

    accuracy                           0.85       848
   macro avg       0.68      0.53      0.52       848
weighted avg       0.80      0.85      0.80       848


Confusion Matrix:
 [[710   9]
 [120   9]]

Model trained and saved successfully!


In [18]:
import pandas as pd
import pickle
import gradio as gr

# Load model and feature columns
with open("heart_disease_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("heart_disease_columns.pkl", "rb") as f:
    columns = pickle.load(f)

# Load dataset for default values
df = pd.read_csv("data.csv")
defaults = df[columns].median(numeric_only=True).to_dict()

# Prediction function
def predict_heart_disease(
    male, age, education, currentSmoker, cigsPerDay,
    BPMeds, prevalentStroke, prevalentHyp, diabetes,
    totChol, sysBP, diaBP, BMI, heartRate, glucose
):
    values = {
        "male": male,
        "age": age,
        "education": education,
        "currentSmoker": currentSmoker,
        "cigsPerDay": cigsPerDay,
        "BPMeds": BPMeds,
        "prevalentStroke": prevalentStroke,
        "prevalentHyp": prevalentHyp,
        "diabetes": diabetes,
        "totChol": totChol,
        "sysBP": sysBP,
        "diaBP": diaBP,
        "BMI": BMI,
        "heartRate": heartRate,
        "glucose": glucose
    }

    input_df = pd.DataFrame([values])[columns]
    prediction = int(model.predict(input_df)[0])
    probability = model.predict_proba(input_df)[0][1] * 100

    result = (
        "Positive prediction for 10-year CHD outcome"
        if prediction == 1
        else "Negative prediction for 10-year CHD outcome"
    )

    return f"{result}\nModel-estimated positive-class probability: {probability:.2f}%"

# Create Gradio interface
inputs = [
    gr.Radio([0, 1], label="Male (1=Yes, 0=No)", value=0),
    gr.Number(label="Age", value=defaults.get("age", 50)),
    gr.Number(label="Education", value=defaults.get("education", 1)),
    gr.Radio([0, 1], label="Current Smoker", value=0),
    gr.Number(label="Cigarettes Per Day", value=defaults.get("cigsPerDay", 0)),
    gr.Radio([0, 1], label="Blood Pressure Medication", value=0),
    gr.Radio([0, 1], label="Prevalent Stroke", value=0),
    gr.Radio([0, 1], label="Prevalent Hypertension", value=0),
    gr.Radio([0, 1], label="Diabetes", value=0),
    gr.Number(label="Total Cholesterol", value=defaults.get("totChol", 200)),
    gr.Number(label="Systolic Blood Pressure", value=defaults.get("sysBP", 120)),
    gr.Number(label="Diastolic Blood Pressure", value=defaults.get("diaBP", 80)),
    gr.Number(label="BMI", value=defaults.get("BMI", 25)),
    gr.Number(label="Heart Rate", value=defaults.get("heartRate", 75)),
    gr.Number(label="Glucose", value=defaults.get("glucose", 80))
]

demo = gr.Interface(
    fn=predict_heart_disease,
    inputs=inputs,
    outputs=gr.Textbox(label="Prediction Result"),
    title="Heart Disease Prediction",
    description="Predict the 10-year CHD outcome using Logistic Regression."
)

demo.launch()

# Code End


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import pandas as pd
import pickle
from flask import Flask, request, render_template_string

# Load model and feature columns
with open("heart_disease_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("heart_disease_columns.pkl", "rb") as f:
    columns = pickle.load(f)

app = Flask(__name__)

labels = {
    "male": "Male (1=Yes, 0=No)",
    "age": "Age",
    "education": "Education",
    "currentSmoker": "Current Smoker",
    "cigsPerDay": "Cigarettes Per Day",
    "BPMeds": "Blood Pressure Medication",
    "prevalentStroke": "Previous Stroke",
    "prevalentHyp": "Prevalent Hypertension",
    "diabetes": "Diabetes",
    "totChol": "Total Cholesterol",
    "sysBP": "Systolic Blood Pressure",
    "diaBP": "Diastolic Blood Pressure",
    "BMI": "BMI",
    "heartRate": "Heart Rate",
    "glucose": "Glucose"
}

binary_fields = [
    "male", "currentSmoker", "BPMeds",
    "prevalentStroke", "prevalentHyp", "diabetes"
]

df = pd.read_csv("data.csv")
defaults = df[columns].median(numeric_only=True).to_dict()

HTML = """
<!DOCTYPE html>
<html>
<head>
<title>Heart Disease Prediction</title>
<style>
body {font-family:Arial;background:#f2f5f9;padding:25px;}
.container {max-width:800px;margin:auto;background:white;padding:25px;border-radius:10px;}
h1 {text-align:center;color:#173b63;}
.grid {display:grid;grid-template-columns:1fr 1fr;gap:15px;}
label {display:block;font-weight:bold;margin-bottom:6px;}
input,select {width:100%;padding:10px;box-sizing:border-box;}
button {width:100%;padding:12px;margin-top:20px;background:#1769aa;color:white;border:0;cursor:pointer;}
.result {margin-top:20px;padding:15px;background:#e6f5e9;text-align:center;}
@media(max-width:600px){.grid{grid-template-columns:1fr;}}
</style>
</head>
<body>
<div class="container">
<h1>Heart Disease Prediction</h1>
<form method="POST">
<div class="grid">
{% for field in columns %}
<div>
<label>{{ labels[field] }}</label>
{% if field in binary_fields %}
<select name="{{ field }}" required>
<option value="0" {% if values.get(field, '0')|string == '0' %}selected{% endif %}>0 - No</option>
<option value="1" {% if values.get(field, '0')|string == '1' %}selected{% endif %}>1 - Yes</option>
</select>
{% else %}
<input type="number" step="any" min="0" name="{{ field }}"
value="{{ values.get(field, defaults.get(field, 0)) }}" required>
{% endif %}
</div>
{% endfor %}
</div>
<button type="submit">Predict</button>
</form>

{% if result %}
<div class="result">
<h3>{{ result }}</h3>
<p>Model-estimated positive-class probability: {{ probability }}%</p>
</div>
{% endif %}

{% if error %}
<p style="color:red;">{{ error }}</p>
{% endif %}
</div>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    values = {}
    result = None
    probability = None
    error = None

    if request.method == "POST":
        try:
            values = request.form.to_dict()
            input_data = {
                col: float(values[col]) for col in columns
            }
            input_df = pd.DataFrame([input_data], columns=columns)

            prediction = int(model.predict(input_df)[0])
            probability = round(
                float(model.predict_proba(input_df)[0][1]) * 100, 2
            )

            result = (
                "Positive prediction for 10-year CHD outcome"
                if prediction == 1
                else "Negative prediction for 10-year CHD outcome"
            )
        except Exception as e:
            error = str(e)

    return render_template_string(
        HTML,
        columns=columns,
        labels=labels,
        binary_fields=binary_fields,
        defaults=defaults,
        values=values,
        result=result,
        probability=probability,
        error=error
    )

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5004, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5004
Press CTRL+C to quit
127.0.0.1 - - [12/Oct/2026 00:35:20] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Oct/2026 00:35:26] "POST / HTTP/1.1" 200 -
